<a href="https://colab.research.google.com/github/Prajwaln08/SQLSmith/blob/main/notebooks/08_QLoRA_SFT_1_5B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQLSmith — Notebook 08: QLoRA SFT (Qwen2.5-Coder-1.5B)

**Milestone B | T4 GPU | Spider training-only supervision**

This notebook fine-tunes a 4-bit Qwen2.5-Coder-1.5B-Instruct model using **LoRA adapters** with loss applied **only to assistant SQL tokens**. Training data come solely from Notebook 07. The reserved Spider development set is **not** used for training or checkpoint selection.

**Execution:** run in order. Begin with `MAX_STEPS=30` to smoke-test; set `MAX_STEPS=250` for a first experiment, then restart from an existing `checkpoint-*` to extend. The reported loss is **not SQL execution accuracy**. Notebook 10 will compare base and adapted models using a frozen Spider evaluator.

## 1 — Install dependencies

After installing, if Colab indicates a restart is required, restart the session and rerun the setup cells. Pin and record versions for final reproducibility.

In [1]:
%pip install -q -U \
    "bitsandbytes>=0.46.1" \
    transformers \
    accelerate \
    peft \
    trl \
    datasets \
    safetensors

In [2]:
import torch
import bitsandbytes as bnb
import transformers
import importlib.metadata as metadata

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Bitsandbytes:", metadata.version("bitsandbytes"))
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), "Enable T4 GPU in Colab"

print("GPU:", torch.cuda.get_device_name(0))
print("Environment verification passed.")

PyTorch: 2.14.1+cu130
Transformers: 5.24.0
Bitsandbytes: 0.50.2
CUDA available: True
GPU: Tesla T4
Environment verification passed.


## 2 — Mount Drive; configure paths and experiment

In [3]:
from google.colab import drive
from pathlib import Path
import os, json, random, time, hashlib, shutil, platform, importlib.metadata
import numpy as np
import torch

drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/Prajwal/Projects/SQLSmith')
DATA_DIR = PROJECT / 'data/processed/sft_v1'
OUT_DIR = PROJECT / 'models/qlora_1p5b_sft_v1'
REPORT_DIR = PROJECT / 'results/milestone_B'
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_FILE = DATA_DIR / 'train_chat.jsonl'
VAL_FILE = DATA_DIR / 'val_chat.jsonl'
MANIFEST = DATA_DIR / 'dataset_manifest.json'
for f in (TRAIN_FILE, VAL_FILE, MANIFEST):
    assert f.is_file(), f'Missing Notebook 07 artifact: {f}'
MODEL_ID = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
SEED = 42
MAX_SEQ_LEN = 1536
MAX_STEPS = 250      # Set 30 for a quick first smoke test
VAL_EXAMPLES = 128  # Fixed, database-stratified internal validation subset
SAVE_STEPS = 50
EVAL_STEPS = 50
LR = 2e-4

assert torch.cuda.is_available(), 'Enable T4 GPU under Runtime > Change runtime type'
print('GPU:', torch.cuda.get_device_name(0))
print('Available VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def sha256_file(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()
print('Training file SHA:', sha256_file(TRAIN_FILE))
print('Validation file SHA:', sha256_file(VAL_FILE))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4
Available VRAM GiB: 14.56
Training file SHA: 8d616419aebed7ddaf55f1ae02331ea041c76f34b732219ed7ba37493cc504ba
Validation file SHA: 474a0680030b288d8d66b2aafae5b93d2ddd6c3793a1dbc59632703772c7a9ab


## 3 — Load and validate Notebook 07 data

Use **training split** for optimizer updates and **internal validation split** for selection. No Spider development examples are read.

In [4]:
def read_jsonl(path):
    with Path(path).open(encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]
train_rows=read_jsonl(TRAIN_FILE)
val_rows=read_jsonl(VAL_FILE)
assert len(train_rows)==7931 and len(val_rows)==717, (len(train_rows), len(val_rows))
assert all(len(x['messages'])==3 and x['messages'][-1]['role']=='assistant' for x in train_rows+val_rows)
train_db={x['db_id'] for x in train_rows}
val_db={x['db_id'] for x in val_rows}
assert train_db.isdisjoint(val_db), 'Validation database leakage!'
assert len(train_db)==131 and len(val_db)==15
ids_tr={x['id'] for x in train_rows}; ids_val={x['id'] for x in val_rows}
assert ids_tr.isdisjoint(ids_val)
with MANIFEST.open() as f: manifest=json.load(f)
print('Train:',len(train_rows),'DBs:',len(train_db))
print('Validation:',len(val_rows),'DBs:',len(val_db))
print('Manifest keys:',list(manifest)[:20])
print('Example roles:',[m['role'] for m in train_rows[0]['messages']])

Train: 7931 DBs: 131
Validation: 717 DBs: 15
Manifest keys: ['project', 'stage', 'dataset', 'source_repository', 'source_archive_sha256', 'sources', 'seed', 'validation_fraction_of_database_ids', 'train_database_ids', 'validation_database_ids', 'train_examples', 'validation_examples', 'excluded_count', 'exact_duplicate_count', 'gold_sql_quarantine_count', 'gold_sql_rejection_summary', 'spider_dev_database_ids_excluded', 'system_prompt', 'validation_design', 'format']
Example roles: ['system', 'user', 'assistant']


## 4 — Fixed internal validation sample, balanced across databases

Keeping one constant sample makes checkpoint comparisons consistent and reduces GPU evaluation overhead.

In [5]:
from collections import defaultdict
by_db=defaultdict(list)
for item in val_rows: by_db[item['db_id']].append(item)
for v in by_db.values():v.sort(key=lambda x:x['id'])
chosen=[]
for db_id in sorted(by_db):
    chosen.append(by_db[db_id][0])
remaining=VAL_EXAMPLES-len(chosen)
assert remaining>=0
rest=[row for db_id in sorted(by_db) for row in by_db[db_id][1:]]
rng=random.Random(SEED)
rng.shuffle(rest)
chosen+=rest[:remaining]
assert len(chosen)==VAL_EXAMPLES
val_sample_ids=[r['id'] for r in chosen]
print('Internal validation examples selected:',len(chosen), 'from DBs:',len({r['db_id'] for r in chosen}))

Internal validation examples selected: 128 from DBs: 15


## 5 — Load 4-bit model + tokenizer

This follows Hugging Face's QLoRA preparation approach: quantize base weights to NF4, freeze base model weights, and train LoRA adapters. T4 uses float16 instead of bfloat16.

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

quant=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=quant, device_map={'':0}, dtype=torch.float16
)
model.config.use_cache=False
model=prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lora_config=LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    task_type='CAUSAL_LM',
)
model=get_peft_model(model,lora_config)
model.print_trainable_parameters()
print('GPU allocated GiB:',round(torch.cuda.memory_allocated()/2**30,3))

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
GPU allocated GiB: 1.578


## 6 — Tokenize with assistant-only loss

We explicitly mask the system and user tokens with `-100`. This prevents the model from being rewarded for reproducing the question or schema. Oversized examples are **excluded**, never silently truncated. Review the excluded count before proceeding.

In [7]:
from datasets import Dataset

def tokenize_row(row):
    messages=row['messages']
    prompt=tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
    response=messages[-1]['content']
    prompt_ids=tokenizer(prompt,add_special_tokens=False)['input_ids']
    response_ids=tokenizer(response,add_special_tokens=False)['input_ids']
    if tokenizer.eos_token_id is not None: response_ids=response_ids+[tokenizer.eos_token_id]
    input_ids=prompt_ids+response_ids
    if len(input_ids)>MAX_SEQ_LEN or not response_ids:return None
    return {'input_ids':input_ids,'attention_mask':[1]*len(input_ids),
            'labels':[-100]*len(prompt_ids)+response_ids,
            'id':row['id'],'db_id':row['db_id'],
            'length':len(input_ids),'target_length':len(response_ids)}

train_encoded=[tokenize_row(r) for r in train_rows]
val_encoded=[tokenize_row(r) for r in chosen]
num_train_dropped=sum(x is None for x in train_encoded)
num_val_dropped=sum(x is None for x in val_encoded)
train_encoded=[x for x in train_encoded if x is not None]
val_encoded=[x for x in val_encoded if x is not None]
assert train_encoded and val_encoded
assert all(any(y!=-100 for y in x['labels']) for x in train_encoded+val_encoded)
print('Training retained:',len(train_encoded),'dropped:',num_train_dropped)
print('Validation retained:',len(val_encoded),'dropped:',num_val_dropped)
print('Train max length:',max(x['length'] for x in train_encoded))
print('Assistant target lengths, first 5:',[x['target_length'] for x in train_encoded[:5]])
train_dataset=Dataset.from_list([{k:v for k,v in x.items() if k in ('input_ids','attention_mask','labels')} for x in train_encoded])
eval_dataset=Dataset.from_list([{k:v for k,v in x.items() if k in ('input_ids','attention_mask','labels')} for x in val_encoded])

Training retained: 7849 dropped: 82
Validation retained: 128 dropped: 0
Train max length: 894
Assistant target lengths, first 5: [14, 15, 14, 20, 19]


## 7 — Dynamic padding and a loss-mask check

Do not substitute `DataCollatorForLanguageModeling`: that would overwrite our target masks.

In [8]:
from dataclasses import dataclass

@dataclass
class AssistantOnlyCollator:
    pad_id:int
    pad_to_multiple_of:int=8
    def __call__(self,features):
        length=max(len(f['input_ids']) for f in features)
        length=((length+self.pad_to_multiple_of-1)//self.pad_to_multiple_of)*self.pad_to_multiple_of
        out={'input_ids':[],'attention_mask':[],'labels':[]}
        for f in features:
            pad=length-len(f['input_ids'])
            out['input_ids'].append(f['input_ids']+[self.pad_id]*pad)
            out['attention_mask'].append(f['attention_mask']+[0]*pad)
            out['labels'].append(f['labels']+[-100]*pad)
        return {k:torch.tensor(v,dtype=torch.long) for k,v in out.items()}

collator=AssistantOnlyCollator(tokenizer.pad_token_id)
batch=collator([train_dataset[0],train_dataset[1]])
assert batch['input_ids'].shape==batch['labels'].shape
assert (batch['labels']!=-100).sum().item()>0
assert torch.all(batch['labels'][batch['attention_mask']==0]==-100)
print('Collated shape:',tuple(batch['input_ids'].shape))
print('Target tokens:',int((batch['labels']!=-100).sum()))

Collated shape: (2, 184)
Target tokens: 29


## 8 — Training configuration

Run a **30-step smoke test** first by setting `MAX_STEPS=30` in Section 2. To extend a prior run, increase `MAX_STEPS` and use the resume section. Evaluations are token-level validation loss, **not SQL execution accuracy**.

In [11]:
from transformers import Trainer, TrainingArguments

training_args=TrainingArguments(
    output_dir=str(OUT_DIR/'trainer_checkpoints'),
    max_steps=MAX_STEPS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_steps=max(1, int(MAX_STEPS * 0.03)),
    weight_decay=0.0,
    optim='paged_adamw_8bit',
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    eval_strategy='steps',
    eval_steps=EVAL_STEPS,
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    logging_steps=10,
    report_to='none',
    remove_unused_columns=False,
    seed=SEED,
    data_seed=SEED,
    dataloader_num_workers=0,
    max_grad_norm=0.3,
)
trainer=Trainer(model=model,args=training_args,train_dataset=train_dataset,
                eval_dataset=eval_dataset,data_collator=collator)
print('Training config ready; max_steps:',MAX_STEPS)
print('Checkpoint root:',training_args.output_dir)

/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)


Training config ready; max_steps: 250
Checkpoint root: /content/drive/MyDrive/Prajwal/Projects/SQLSmith/models/qlora_1p5b_sft_v1/trainer_checkpoints


## 9 — Train or resume

Checkpoints include optimizer and scheduler state. **Do not use `trainer.train(resume_from_checkpoint=True)` unless a checkpoint exists.** Drive can be slow; keep the notebook connected while the training cell is running. Do not delete old checkpoints until exports are verified.

In [12]:
from transformers.trainer_utils import get_last_checkpoint
checkpoint_root=Path(training_args.output_dir)
checkpoint_root.mkdir(parents=True, exist_ok=True)
last_checkpoint=get_last_checkpoint(str(checkpoint_root))
print('Resuming:',last_checkpoint or 'No checkpoint — fresh training')
started=time.perf_counter()
train_result=trainer.train(resume_from_checkpoint=last_checkpoint)
training_seconds=time.perf_counter()-started
print('Training cell finished. Elapsed seconds:',round(training_seconds,1))
print('Final training metrics:',train_result.metrics)

Resuming: No checkpoint — fresh training


Step,Training Loss,Validation Loss
50,0.204627,0.225545


Step,Training Loss,Validation Loss
50,0.204627,0.225545
100,0.198842,0.215909
150,0.173390,0.209042
200,0.159680,0.198297
250,0.163304,0.202256


Training cell finished. Elapsed seconds: 1542.8
Final training metrics: {'train_runtime': 1541.0167, 'train_samples_per_second': 1.298, 'train_steps_per_second': 0.162, 'total_flos': 5605347554574336.0, 'train_loss': 0.19123511219024658, 'epoch': 0.25480952987641736}


## 10 — Evaluate internal validation loss and export adapter

In [13]:
validation_metrics=trainer.evaluate()
print('Internal validation metrics:',validation_metrics)
ADAPTER_DIR=OUT_DIR/'final_adapter'
ADAPTER_DIR.mkdir(parents=True,exist_ok=True)
trainer.model.save_pretrained(str(ADAPTER_DIR),safe_serialization=True)
tokenizer.save_pretrained(str(ADAPTER_DIR))
assert (ADAPTER_DIR/'adapter_config.json').exists()
assert any(ADAPTER_DIR.glob('adapter_model.*'))
print('Adapter exported:',ADAPTER_DIR)
print('This is only a LoRA adapter, not a merged standalone model.')

Training Loss,Validation Loss,Step
0.163304,0.202256,250


Internal validation metrics: {'eval_loss': 0.20225580036640167}
Adapter exported: /content/drive/MyDrive/Prajwal/Projects/SQLSmith/models/qlora_1p5b_sft_v1/final_adapter
This is only a LoRA adapter, not a merged standalone model.


## 11 — Save metadata and checksums

These outputs will be used by Notebook 10. Keep the original Spider development-set predictions untouched.

In [14]:
from datetime import datetime, timezone

versions={p:importlib.metadata.version(p) for p in ['torch','transformers','accelerate','peft','bitsandbytes','datasets']}
metrics={k:float(v) if isinstance(v,(float,int,np.floating,np.integer)) else str(v)
         for k,v in validation_metrics.items()}
report={
 'project':'SQLSmith','notebook':'08_QLoRA_SFT_1_5B','milestone':'B',
 'model_id':MODEL_ID,'method':'QLoRA NF4 + LoRA r16, assistant SQL-only cross-entropy',
 'train_original':len(train_rows),'validation_original':len(val_rows),
 'train_accepted':len(train_encoded),'train_excluded_length':num_train_dropped,
 'internal_validation_sample_selected':len(chosen),'internal_validation_accepted':len(val_encoded),
 'validation_excluded_length':num_val_dropped,
 'train_db_count':len(train_db),'validation_db_count':len(val_db),
 'seed':SEED,'max_seq_len':MAX_SEQ_LEN,'max_steps':MAX_STEPS,'learning_rate':LR,
 'train_jsonl_sha256':sha256_file(TRAIN_FILE),'val_jsonl_sha256':sha256_file(VAL_FILE),
 'validation_example_ids':val_sample_ids,
 'adapter_dir':str(ADAPTER_DIR),'metrics':metrics,
 'training_seconds':round(training_seconds,2),
 'versions':versions,'gpu':torch.cuda.get_device_name(0),
 'peak_gpu_memory_allocated_gib':round(torch.cuda.max_memory_allocated()/2**30,3),
 'spider_dev_used_for_training':False,'official_test_suite_accuracy':None,
 'created_utc':datetime.now(timezone.utc).isoformat()
}
report_path=REPORT_DIR/'08_qlora_1p5b_train_report.json'
report_path.write_text(json.dumps(report,indent=2),encoding='utf-8')
print('Report:',report_path)
print('Max GPU allocated GiB:',report['peak_gpu_memory_allocated_gib'])
print('Adapter files:',[p.name for p in ADAPTER_DIR.iterdir()])

Report: /content/drive/MyDrive/Prajwal/Projects/SQLSmith/results/milestone_B/08_qlora_1p5b_train_report.json
Max GPU allocated GiB: 3.767
Adapter files: ['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json']


## 12 — Final QA and next steps

- This notebook does **not** claim a new Spider execution score.
- Notebook **09** will apply an analogous QLoRA protocol to the 7B model (with tighter T4 memory controls).
- Notebook **10** will compare both fine-tuned models against frozen base-model predictions, using the official Spider test-suite evaluator and consistent inference settings.
- Only the notebook, configuration and non-sensitive reports should be committed to public GitHub; store adapters and checkpoints in Drive or an approved model registry.

In [15]:
assert report_path.exists() and ADAPTER_DIR.exists()
assert not train_db.intersection(val_db)
assert len(train_encoded)>0 and len(val_encoded)>0
print('PASS: adapter + training manifest saved')
print('PASS: no training/validation database overlap')
print('PASS: Spider dev is not loaded in this notebook')

PASS: adapter + training manifest saved
PASS: no training/validation database overlap
PASS: Spider dev is not loaded in this notebook
